In [ ]:
import modal

image = (
    modal.Image.debian_slim(python_version="3.14")
    .uv_pip_install(
        "torch",
        "transformers",
        "trl",
        "peft",
        "accelerate",
        "datasets",
        "scikit-learn",
    )
)

app = modal.App("base_vs_dpo_vs_sft", image=image)

hf_secret = modal.Secret.from_name("huggingface")

In [ ]:
from datasets import load_dataset, DatasetDict

def build_dataset(train_size=20_000, val_size=1_000):
    split = load_dataset("trl-lib/ultrafeedback_binarized", split="train") \
        .shuffle(seed=42) \
        .train_test_split(test_size=0.1)

    ds = DatasetDict({
        "train": split["train"].select(range(train_size)),
        "val": split["test"].select(range(val_size)),
        "test": load_dataset("trl-lib/ultrafeedback_binarized", split="test"),
    })
    ds = ds.select_columns(["chosen", "rejected"])
    return ds

In [ ]:
import torch
from trl import DPOTrainer, DPOConfig
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, PeftModel

REWARD_MODEL_REPO = "hyerra/qwen3-4b-reward-ultrafeedback"


@app.function(gpu="A100", secrets=[hf_secret], timeout=60 * 60 * 8)
def compare_base_sft_dpo():
    ds = build_dataset()

    model_name = "Qwen/Qwen2.5-1.5B"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    instruction_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
    instruction_tokenizer = AutoTokenizer.from_pretrained(instruction_model_name)

    if instruction_tokenizer.pad_token is None:
        instruction_tokenizer.pad_token = instruction_tokenizer.eos_token

    dpo_model = AutoModelForCausalLM.from_pretrained(
        instruction_model_name,
        dtype=torch.bfloat16,
    ).to("cuda")

    trainer = DPOTrainer(
        dpo_model,
        args=DPOConfig(
            bf16=True,
            num_train_epochs=1,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            max_length=1024,
            eval_strategy="steps",
            save_strategy="steps",
            eval_steps=500,
            save_steps=500,
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
        ),
        train_dataset=ds["train"],
        eval_dataset=ds["val"],
        processing_class=instruction_tokenizer,
        peft_config=LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            task_type="CAUSAL_LM",
        )
    )
    trainer.train()

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16,
    ).to("cuda")

    sft_model = AutoModelForCausalLM.from_pretrained(
        instruction_model_name,
        dtype=torch.bfloat16,
    ).to("cuda")

    rm_base = AutoModelForSequenceClassification.from_pretrained(
        "Qwen/Qwen3-4B",
        num_labels=1,
        dtype=torch.bfloat16,
    )
    rm = PeftModel.from_pretrained(rm_base, REWARD_MODEL_REPO).to("cuda").eval()
    rm_tokenizer = AutoTokenizer.from_pretrained(REWARD_MODEL_REPO)

    if rm_tokenizer.pad_token is None:
        rm_tokenizer.pad_token = rm_tokenizer.eos_token
    rm.config.pad_token_id = rm_tokenizer.pad_token_id

    @torch.no_grad()
    def evaluate(conversations, model, tokenizer, reward_model, reward_tokenizer, chat_template_gen=False, batch_size=32):
        model.eval()
        prompts = [convo[:-1] for convo in conversations["chosen"]]

        rewards = []
        for i in range(0, len(prompts), batch_size):
            batch = prompts[i:i+batch_size]
            if chat_template_gen:
                inputs = tokenizer.apply_chat_template(
                    batch,
                    tokenize=True,
                    padding=True,
                    padding_side="left",
                    add_generation_prompt=True,
                    return_tensors="pt",
                    return_dict=True
                ).to(model.device)
            else:
                raw_batch = ['\n'.join([f"{msg["role"]}: {msg["content"]}" for msg in convo]) for convo in batch]
                inputs = tokenizer(raw_batch, padding=True, padding_side="left", return_tensors="pt").to(model.device)
            out = model.generate(**inputs, max_new_tokens=256)
            new_tokens = out[:, inputs["input_ids"].shape[1]:]
            responses = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

            scored = [conv + [{"role": "assistant", "content": resp}] for conv, resp in zip(batch, responses)]

            rm_in = reward_tokenizer.apply_chat_template(
                scored, tokenize=True, padding=True, add_generation_prompt=False, return_tensors="pt", return_dict=True
            ).to(reward_model.device)
            rewards.append(reward_model(**rm_in).logits.squeeze(-1))
        return torch.cat(rewards).mean().item()

    base_score = evaluate(ds["test"], model, tokenizer, rm, rm_tokenizer, chat_template_gen=False)
    sft_score = evaluate(ds["test"], sft_model, instruction_tokenizer, rm, rm_tokenizer, chat_template_gen=True)
    dpo_score = evaluate(ds["test"], trainer.model, instruction_tokenizer, rm, rm_tokenizer, chat_template_gen=True)

    return base_score, sft_score, dpo_score

In [ ]:
with app.run():
    base_score, sft_score, dpo_score = compare_base_sft_dpo.remote()
print(f"Base: {base_score}, SFT: {sft_score}, DPO: {dpo_score}")